# Imports

In [10]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import sklearn
import numpy as np
import pandas as pd
import pickle

from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
# from utils_for_notebooks import read_per_model_info
from scripts.evaluate_mmlu import evaluate_mmlu
sys.path.pop(0);


CACHE_DIR = "./cache_dir"


# Functions

In [35]:
def openllmname_to_hf_model_id(openllm_name):
    assert openllm_name.startswith(
        "open-llm-leaderboard/"
    ), "OpenLLM name must start with 'open-llm-leaderboard/'"
    post_details = openllm_name.split("open-llm-leaderboard/details_")[1]
    creator, model = post_details.split("__")
    hf_model_id = f"{creator}/{model}"
    return hf_model_id


def read_per_model_info(model_id, timestamp, scenario, cache_dir=CACHE_DIR):
    # model_id = "meta-llama/Llama-2-13b-hf"
    creator, model = tuple(model_id.split("/"))
    model_details_id = "open-llm-leaderboard/details_{:}__{:}".format(
        creator, model
    )

    # s = "harness_hendrycksTest_abstract_algebra_5"
    s = scenario
    aux = load_dataset(model_details_id, s, cache_dir=cache_dir)
    print("Available timestamps:")
    print(aux.keys())

    # Attempt to create a table with all available columns.
    latest_data = aux[timestamp]

    # The structure may be a dict with lists as values, or list of dicts.
    # We need to check what we have.

    if isinstance(latest_data, dict):
        # If values are lists of the same length, treat as column-wise.
        lens = [len(v) for v in latest_data.values() if isinstance(v, list)]
        if lens and all(l == lens[0] for l in lens):
            # Looks like column-wise dict of lists.
            df = pd.DataFrame(latest_data)
        else:
            # Dict where each key might be a scalar or something else.
            df = pd.DataFrame([latest_data])
    else:
        # Try to coerce into DataFrame anyway
        df = pd.DataFrame(latest_data)

    # print(df)
    return df


def compute_accuracy(df, metric_key="acc"):
    # Compute accuracy as number of rows where df["metrics"]["acc"] == 1
    # Each element in df["metrics"] is likely a dict with an 'acc' key
    if "metrics" in df.columns:
        num_correct_acc = sum(1 for metric in df["metrics"] if metric.get('acc') == 1)
        num_correct_norm_acc = sum(1 for metric in df["metrics"] if metric.get('norm_acc') == 1)

    else:
        num_correct_acc = sum(1 for acc in df["acc"] if acc == 1)
        num_correct_norm_acc = sum(1 for acc in df["acc_norm"] if acc == 1)

    acc = num_correct_acc / len(df)
    norm_acc = num_correct_norm_acc / len(df)
    print(f"Accuracy (number of acc == 1): {num_correct_acc} / {len(df)}")
    print(f"Accuracy (number of acc == 1): {acc}")
    print(f"Accuracy (number of norm_acc == 1): {num_correct_norm_acc} / {len(df)}")
    print(f"Accuracy (number of norm_acc == 1): {norm_acc}")

    return acc, norm_acc

# Analyze

## Save questions

In [54]:
model_df_0819 = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    # timestamp='2023_08_19T22_35_38.117975',
    timestamp='latest',
    scenario="harness_hendrycksTest_anatomy_5",
    cache_dir=CACHE_DIR
)
compute_accuracy(model_df_0819)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])
Accuracy (number of acc == 1): 69 / 135
Accuracy (number of acc == 1): 0.5111111111111111
Accuracy (number of norm_acc == 1): 69 / 135
Accuracy (number of norm_acc == 1): 0.5111111111111111


(0.5111111111111111, 0.5111111111111111)

In [51]:
model_df_0819

,acc,acc_norm,choices,cont_tokens,example,full_prompt,gold,hashes,input_tokens,instruction,num_asked_few_shots,num_effective_few_shots,padded,predictions,query,truncated
0,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Gastrulation is the process of\nA. mesoderm fo...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[133, 133, 133, 133]","[-1.712890625, -1.275390625, -1.087890625, -1....",Gastrulation is the process of\nA. mesoderm fo...,"[0, 0, 0, 0]"
1,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Blood flows from the right ventricle of the he...,The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[144, 144, 144, 144]","[-1.99609375, -1.87109375, -1.43359375, -0.777...",Blood flows from the right ventricle of the he...,"[0, 0, 0, 0]"
2,1.0,1.0,"[A, B, C, D]","[[319], [350], [315], [360]]","In men, specimens for gonococcal cultures are ...",The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[150, 150, 150, 150]","[-2.69140625, -2.41015625, -0.55029296875, -1....","In men, specimens for gonococcal cultures are ...","[0, 0, 0, 0]"
3,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Primary motor cortex activity results in\nA. b...,The following are multiple choice questions (w...,3,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[125, 125, 125, 125]","[-1.431640625, -2.041015625, -0.80712890625, -...",Primary motor cortex activity results in\nA. b...,"[0, 0, 0, 0]"
4,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Saliva contains an enzyme that acts upon which...,The following are multiple choice questions (w...,0,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[158, 158, 158, 158]","[-1.923828125, -0.314208984375, -2.892578125, ...",Saliva contains an enzyme that acts upon which...,"[0, 0, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,1.0,1.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Which of the following best describes the proc...,The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[145, 145, 145, 145]","[-5.9140625, -4.921875, -0.0248870849609375, -...",Which of the following best describes the proc...,"[0, 0, 0, 0]"
131,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",A possible effect of damage to the third crani...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[141, 141, 141, 141]","[-0.9423828125, -1.8173828125, -1.5048828125, ...",A possible effect of damage to the third crani...,"[0, 0, 0, 0]"
132,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",The major concentrations of proprioceptive rec...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[79, 79, 79, 79]","[-0.94921875, -1.76171875, -1.66796875, -1.402...",The major concentrations of proprioceptive rec...,"[0, 0, 0, 0]"
133,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Which of the following anatomical regions of a...,The following are multiple choice questions (w...,0,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[147, 147, 147, 147]","[-1.40234375, -1.32421875, -1.41796875, -1.464...",Which of the following anatomical regions of a...,"[0, 0, 0, 0]"


In [53]:
import json

# Extract the relevant columns from model_df_0819
data_to_save = []
for idx, row in model_df_0819.iterrows():
    entry = {
        "example": row.get("example"),
        "full_prompt": row.get("full_prompt"),
        "query": row.get("query"),
        "choices": row.get("choices"),
        "gold": row.get("gold"),
    }
    data_to_save.append(entry)

# Save to json file
with open("anatomy_prompts_examples.json", "w") as outfile:
    json.dump(data_to_save, outfile, indent=2)

print(f"Saved {len(data_to_save)} records with 'example', 'full_prompt', and 'query' to anatomy_prompts_examples.json.")


Saved 135 records with 'example', 'full_prompt', and 'query' to anatomy_prompts_examples.json.


## Compare results from outputs and hf

In [28]:
with open("/home/oh/arubinstein17/github/disco-public/cache/target_outputs2.pkl", "rb") as f:
    target_outputs = pickle.load(f)


In [29]:
print(target_outputs.keys())
print(target_outputs["predictions"].shape)
print(target_outputs["correctness"].shape)
print(target_outputs["Models"])



dict_keys(['predictions', 'correctness', 'Models', 'Datapoints', 'Scenarios'])
(40, 14042, 31)
(40, 14042, 1)
{'open-llm-leaderboard/details_abacusai__MetaMath-bagel-34b-v0.2-c1500': 0, 'open-llm-leaderboard/details_zhengr__MixTAO-7Bx2-MoE-DPO': 1, 'open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full': 2, 'open-llm-leaderboard/details_LoSboccacc__orthogonal-2x7B-base': 3, 'open-llm-leaderboard/details_rombodawg__Leaderboard-killer-MoE_4x7b': 4, 'open-llm-leaderboard/details_moreh__MoMo-70B-lora-1.8.6-DPO': 5, 'open-llm-leaderboard/details_FelixChao__ExtremeDolphin-MoE': 6, 'open-llm-leaderboard/details_rombodawg__Everyone-Coder-4x7b-Base': 7, 'open-llm-leaderboard/details_nfaheem__Marcoroni-7b-DPO-Merge': 8, 'open-llm-leaderboard/details_Swisslex__Mixtral-Orca-v0.1': 9, 'open-llm-leaderboard/details_deepseek-ai__deepseek-moe-16b-base': 10, 'open-llm-leaderboard/details_wang7776__Mistral-7B-Instruct-v0.2-sparsity-20': 11, 'open-llm-leaderboard/details_BAAI__Aquila2-34B':

In [48]:
def acc_from_outputs(target_outputs, target_model_name, scenario_name):

    # target_model_name = 'open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full'
    # scenario_name = "harness_hendrycksTest_abstract_algebra_5"
    abstract_algebra_indices = target_outputs["Scenarios"][scenario_name]
    target_model_id = target_outputs["Models"][target_model_name]
    correctness = target_outputs["correctness"][target_model_id][abstract_algebra_indices]
    acc = sum(correctness) / len(correctness)
    print(f"sum: {sum(correctness)}")
    print(f"len: {len(correctness)}")
    print(f"acc: {acc}")
    # print(target_model_name)
    # print(target_model_id)
    # print(acc)
    return acc

In [41]:
def acc_from_hf(target_model_name, scenario_name, timestamp):

    # scenario_name = "harness_hendrycksTest_abstract_algebra_5"
    # timestamp = '2024_01_16T04_06_55.134598'

    model_name_hf = openllmname_to_hf_model_id(target_model_name)
    model_df = read_per_model_info(
        model_id=model_name_hf,
        timestamp=timestamp,
        scenario=scenario_name,
        cache_dir=CACHE_DIR
    )
    acc, norm_acc = compute_accuracy(model_df)
    return acc


In [44]:
target_model_name = "open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full"
# scenario_name = "harness_hendrycksTest_abstract_algebra_5"
timestamp = '2024_01_16T04_06_55.134598'
for scenario_name in target_outputs["Scenarios"].keys():
    acc_hf = acc_from_hf(target_model_name, scenario_name, timestamp)
    acc_outputs = acc_from_outputs(target_outputs, target_model_name, scenario_name)
    assert acc_hf == acc_outputs, f"Scenario {scenario_name} failed: {acc_hf} != {acc_outputs}"


Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 30 / 100
Accuracy (number of acc == 1): 0.3
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 77 / 135
Accuracy (number of acc == 1): 0.5703703703703704
Accuracy (number of norm_acc == 1): 0 / 135
Accuracy (number of norm_acc == 1): 0.0


AssertionError: Scenario harness_hendrycksTest_anatomy_5 failed: 0.5703703703703704 != 0.5925925925925926

In [56]:
target_model_name = "open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full"

timestamp = '2024_01_16T04_06_55.134598'
# timestamp = '2024_01_16T04_10_47.293422'
# scenario_name = "harness_hendrycksTest_anatomy_5"
scenario_name = "harness_hendrycksTest_abstract_algebra_5"
# scenario_name = "harness_arc_challenge_25"
acc_hf = acc_from_hf(target_model_name, scenario_name, timestamp)
acc_outputs = acc_from_outputs(target_outputs, target_model_name, scenario_name)
assert acc_hf == acc_outputs, f"Scenario {scenario_name} failed: {acc_hf} != {acc_outputs}"

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 30 / 100
Accuracy (number of acc == 1): 0.3
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
sum: [30.]
len: 100
acc: [0.3]
